# Hotspot + wind rotation kernel — theory and visualization

This notebook derives and visualizes an **emission** rotation kernel for a tidally-locked hot Jupiter with:

- an **elliptical hotspot** on the dayside (east-west offset allowed, e.g. the classic advected-hotspot signature),
- a **3-component wind field**: solid-body rotation, a superrotating equatorial jet, and a day-to-night flow confined to high latitudes.

It follows the same conventions as `SolidRotationKernel` and `CitrusRotationKernel` (same sign convention, same `BaseKerMulti` API), but the brightness map and the wind field are now genuinely 2D (longitude *and* latitude dependent), so the kernel is built **numerically** on a surface grid instead of analytically from chord lengths. The generic (geometry-agnostic) pieces of that numerical machinery (`equal_area_latlon_grid`, `bin_weighted_velocities`, ...) are plain module-level functions in `starships.spectrum`, so they can be reused for other kernels later (e.g. a numerical cross-check of `CitrusRotationKernel` itself, or a future transit/transmission terminator-ring kernel).

`HotspotWindRotationKernel` lives in `starships.spectrum`, alongside every other rotation kernel in this package -- this notebook (`tutorials/rotation_kernel_examples/`) is its "Explained" tutorial, the theory and validation record for that class. It started life as a standalone prototype in `starships_analysis/hotspot_wind_kernel/` (kept separate from `starships` while its API was still moving) and was folded in once validated -- see Section 15 for the diagnostic tooling that migration brought with it (`kernel.show(phase)`, matching `CitrusRotationKernel.show()`).

## 1. The general kernel formalism

A rotation/wind kernel is the brightness-weighted distribution of line-of-sight (LOS) velocities over the visible surface of the planet:

$$K(v) = \iint_{\text{visible}} I(\lambda,\varphi)\, \mu(\lambda,\varphi)\, \delta\big(v - v_{\rm los}(\lambda,\varphi)\big) \, d\Omega$$

where $\lambda$ is longitude, $\varphi$ is latitude, $I$ is the local brightness, $\mu$ is the projected-area (Lambertian) weight, and $d\Omega$ is the solid-angle element. This is the direct 2D generalization of Gray's classical rotational-broadening profile, and it is exactly the same idea `CitrusRotationKernel.get_ker` already implements for the special case where $I$ is piecewise-constant in longitude and the wind is solid-body rotation (so $v_{\rm los}$ depends on longitude only) — in that special case the integral collapses to an analytic chord-length calculation (`citrus_to_ker`).

Once $I$ becomes a genuine 2D function (a hotspot island) or the wind field depends on latitude (a jet, a day-night flow), $\lambda$ and $\varphi$ are coupled inside $v_{\rm los}$ and there is no closed-form shortcut left — the integral has to be evaluated as a numerical quadrature. Section 6 below comes back to this point with a concrete literature example of the *same* idea used in a case where it *does* stay analytic.

## 2. Sky-plane geometry

Planet-centered spherical coordinates: latitude $\varphi$, longitude $\lambda$ (increases eastward; $\lambda = 0$ is the substellar meridian, fixed on the planet's surface for a tidally-locked planet). The sub-observer meridian is $\lambda_{\rm obs} = -2\pi(\text{phase} - 0.5)$ (`phase_to_lam_obs`): co-rotation aligns phase=0.5 with the substellar meridian ($\lambda_{\rm obs}=0$, secondary eclipse), and the *sign* of the drift away from that (not just the alignment) is fixed by requiring a prograde, tidally-locked orbit to actually behave like one -- verified by checking that a fixed surface feature drifts toward, and disappears over, the eastern limb as phase increases past 0.5, matching `starships.orbite.rv_theo_t`'s `RV_planet(phase) = +K\sin(2\pi\,\text{phase})` convention (redshift positive). An earlier version of this notebook had this sign backwards -- it is not obvious from symmetry alone and has to be checked against an actual physical expectation, which is also why the line-of-sight formula in Section 3 needs its own explicit sign check, not just an internal consistency check against the pre-existing kernels.

Projecting onto the sky plane (unit sphere, edge-on orbit, zero obliquity — same implicit assumption already used by every kernel in `starships.spectrum`):

$$x = \cos\varphi \, \sin(\lambda - \lambda_{\rm obs}) \qquad y = \sin\varphi \qquad z = \cos\varphi \, \cos(\lambda - \lambda_{\rm obs})$$

with $x$ = east-west on the sky, $y$ = north-south (projection of the spin axis), $z$ = toward the observer. A point is visible when $z>0$, and for emission the projected-area (Lambertian) weight is simply $\mu = z$.

## 3. Line-of-sight velocity of a zonal wind

Restricting every wind component to the *zonal* direction (tangent to circles of latitude — no north-south component, so none of them can break north-south symmetry by themselves, matching what was asked for), the LOS projection of a local eastward speed $v_{\rm zonal}(\lambda,\varphi)$ works out to a compact formula:

$$v_{\rm los}(\lambda,\varphi) = v_{\rm zonal}(\lambda,\varphi) \, \sin(\lambda - \lambda_{\rm obs})$$

with the **redshift-positive** sign convention already flagged in Section 2 -- fixed here together with $\lambda_{\rm obs}$'s sign, since the two bugs happened to partially mask each other (each is independently checkable, but only in combination with a real feature to track; a plain solid-body disk has no way to reveal either one). Two independent physical checks, not just the shape-only consistency check below: (i) a day-to-night flow carrying gas from the visible dayside into the hidden nightside must be redshifted (receding) when viewed at secondary eclipse (Section 14); (ii) this formula's overall shape still has to match the pre-existing kernels once both signs are fixed.

**Shape consistency check.** For solid-body rotation, $v_{\rm zonal} = \Omega R \cos\varphi$, so $v_{\rm los} = \Omega R \cos\varphi \sin(\lambda-\lambda_{\rm obs}) = \Omega x$ — the same `v = omega * x` *shape* already used by `CitrusRotationKernel.get_ker` and the closed form in `SolidRotationKernel.get_ker`. This only checks the functional form (a uniformly-rotating, brightness-symmetric disk can't reveal a sign error), which is exactly why it did not catch the sign bug on its own -- checked numerically in Section 12 below, but that check alone would have passed both before and after the fix.

## 4. Three wind components

The first two are purely *zonal* (tangent to circles of latitude around the spin axis) and sum into a single $v_{\rm zonal}(\lambda,\varphi)$:

1. **Solid-body rotation**: $\Omega_{\rm rot} \, R \cos\varphi$ (tidally-locked value: $\Omega_{\rm rot} = 2\pi / P_{\rm orb}$).
2. **Superrotating equatorial jet**: an extra $\Delta\Omega_{\rm jet} \, R \cos\varphi$, active only for $|\varphi| < \varphi_{\rm jet}$ (2 parameters: $\Delta\Omega_{\rm jet}$, $\varphi_{\rm jet}$).

The third is genuinely different, and needs its own treatment:

3. **Day-to-night flow**: a constant speed $v_{\rm d2n}$ (not scaled by the $\cos\varphi$ lever arm — a translation speed, not a rotation), active only for $|\varphi| > \varphi_{\rm d2n}$ (2 parameters: $v_{\rm d2n}$, $\varphi_{\rm d2n}$), directed away from the substellar *point* along the great circle through it — **not** tangent to a circle of latitude. Those two directions only coincide exactly on the equator; a first version of this model used $v_{\rm d2n}\,{\rm sign}(\lambda)$ in the zonal direction everywhere -- caught as wrong (by inspecting a phase=0.5 sky view, where it produced an unphysical east/west split instead of the expected center-to-limb, radially-symmetric pattern) and fixed by giving this term its own line-of-sight projection, derived alongside the wind-field plots in Section 10 (`los_velocity_from_day_night`).

All the masks above are booleans multiplying their own term, so a disabled component (default parameters) contributes exactly zero without any special-casing in the code.

## 5. Brightness map: the hotspot ellipse

The hotspot is an ellipse in $(\lambda,\varphi)$ space, centered at $(\lambda_{\rm hs}, \varphi_{\rm hs})$ with half-widths $(a_\lambda, a_\varphi)$. Define the normalized elliptical radius:

$$d(\lambda,\varphi) = \sqrt{\left(\frac{\lambda-\lambda_{\rm hs}}{a_\lambda}\right)^2 + \left(\frac{\varphi-\varphi_{\rm hs}}{a_\varphi}\right)^2}$$

($\lambda - \lambda_{\rm hs}$ wrapped to $(-\pi,\pi]$, so the hotspot can straddle the $\pm\pi$ branch cut). Instead of a hard `d <= 1` boundary (which would introduce a brightness discontinuity, and therefore ringing in the kernel after convolution with the instrumental profile), the hot-region weight is a smoothed step:

$$w_{\rm hot}(\lambda,\varphi) = \frac{1}{2}\left(1 - \tanh\frac{d(\lambda,\varphi)-1}{\epsilon}\right)$$

with $\epsilon$ (`hotspot_edge_width`) controlling the softness of the transition. $w_{\rm hot}$ and $w_{\rm cold} = 1 - w_{\rm hot}$ form a strict partition of unity everywhere on the sphere.

## 6. From an analytic integral to a numerical grid

A companion PDF (Florian's kernel-rotation note, Appendix A) works out the *transit* analogue of this problem: the observable there is the planet's effective radius $R_{\rm eff}^2(\nu)$, obtained by integrating $R_p^2(\nu,\theta)$ around the terminator ($\theta$ = angle around the planet's limb). For solid rotation ($v = v_0\cos\theta$), the substitution $x = v_0\cos\theta$ turns that integral into a **convolution** with an analytic kernel:

$$K(x) \propto \frac{1}{\sqrt{1-(x/v_0)^2}}$$

— the same $x = v_0\cos\theta$ substitution used throughout `starships.spectrum`, just applied to a *ring* (uniform weight per angle around the terminator) instead of a *disk* (chord-length weight per position), which is why the transit kernel shape ($1/\sqrt{1-(x/v_0)^2}$, a double-horned profile) differs from the emission uniform-disk kernel shape ($\sqrt{1-(x/v_0)^2}$, used by `SolidRotationKernel`) even though both come from the very same change of variables. The PDF also confirms the trick used for superrotation: split the integral into a **sum of convolutions over latitude bands**, each with its own velocity shift — exactly the same idea Section 4 above uses for the jet.

That trick is a special case of a more general fact: *any* pushforward of a weight measure through a monotonic function $v(\theta)$ has a closed form (via the Jacobian of the substitution). It breaks down the moment $v$ depends on **two** coupled coordinates that can't be integrated out one at a time — exactly our situation, since the hotspot ellipse and the jet/day-night latitude masks make $v_{\rm los}$ and $I$ both functions of $(\lambda,\varphi)$ jointly.

The fallback that always works is the *discrete* version of the same pushforward: sample the surface on a grid, evaluate $(I \cdot \mu \cdot d\Omega, v_{\rm los})$ at every grid point, and histogram-bin the result onto a velocity grid. This is what the module-level functions right before `HotspotWindRotationKernel` in `starships.spectrum` implement, as three reusable, fully generic pieces:

- `equal_area_latlon_grid`: a $(\varphi,\lambda)$ grid with exactly equal cell area (uniform in $\sin\varphi$, not $\varphi$, so poles are not over-sampled) — sphere-specific, reusable for any full-sphere kernel.
- `project_to_sky` / `los_velocity_from_zonal`: the geometry of Sections 2-3 — sphere-specific.
- `bin_weighted_velocities`: a **fully generic** `(v_los, weight) -> kernel` histogram, with no assumption about where the samples came from (a disk grid, a transit ring, an irregular mesh — anything). This is the piece that most directly plays the role of the analytic $\delta$-function integral in Section 1, and the one most likely to be reused as-is for a future kernel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u

from starships.spectrum import (HotspotWindRotationKernel, SolidRotationKernel,
                                 plot_sky_view, phase_to_lam_obs,
                                 los_velocity_from_zonal, los_velocity_from_day_night)

## 7. Example setup — a hot-Jupiter-like planet

Rough WASP-33b-like scale (radius, tidally-locked ~1.2 day rotation). The hotspot is shifted 20 deg east of the substellar point (the classic advected-hotspot direction), the jet is a moderate superrotating equatorial band, and the day-to-night flow only kicks in poleward of 40 deg.

**Note on `omega`/`jet_delta_omega` units**: following the exact convention already used by `SolidRotationKernel`/`CitrusRotationKernel`, these are passed as plain `1/u.s` quantities (`2 * np.pi / period`), *not* `u.rad/u.s` — astropy does not consider the two directly convertible without an explicit equivalency, since it treats the radian specially.

In [ ]:
pl_rad = 1.5 * u.Rjup
period = 1.22 * u.day
omega_rot = 2 * np.pi / period.to('s')
resolution = 70_000

kernel = HotspotWindRotationKernel(
    pl_rad, omega_rot, resolution,
    hotspot_lon=20 * u.deg, hotspot_lat=0 * u.deg,
    hotspot_half_lon=35 * u.deg, hotspot_half_lat=25 * u.deg, hotspot_edge_width=0.2,
    jet_delta_omega=2 * np.pi / (2.5 * u.day).to('s'), jet_half_lat=15 * u.deg,
    day_night_speed=2000 * u.m / u.s, day_night_lat_onset=40 * u.deg,
    n_lat=181, n_lon=361,
)
kernel

## 8. Brightness map

The hot-region weight $w_{\rm hot}(\lambda,\varphi)$ over the full sphere (not just the visible hemisphere at a given phase — the hotspot is fixed in the planet frame).

In [ ]:
phi, lam, w_hot = kernel.brightness_map()

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.pcolormesh(np.degrees(lam), np.degrees(phi), w_hot, shading='auto', cmap='inferno')
ax.axvline(0, color='cyan', linestyle='--', lw=1, label='substellar meridian')
ax.set_xlabel('Longitude [deg] (0 = substellar)')
ax.set_ylabel('Latitude [deg]')
ax.set_title('Hot-region weight $w_{hot}$')
ax.legend(loc='upper right')
fig.colorbar(im, ax=ax, label='$w_{hot}$')
fig.tight_layout()

## 9. Observer's view of the planet across orbital phase

The brightness map above is phase-independent (fixed in the planet frame). What the observer actually sees at a given phase is its projection onto the sky plane, restricted to the visible hemisphere ($z>0$) -- exactly the `project_to_sky` geometry from Section 2, which is also what the kernel integral itself is evaluated on. `plot_sky_view` renders this directly (the same idea as `CitrusRotationKernel.show()`'s `plot_sphere` panel, but for a full 2D map instead of a handful of longitude wedges).

In [ ]:
phases_view = [0.0, 0.15, 0.25, 0.5, 0.75, 0.9]
fig, axes = plt.subplots(1, len(phases_view), figsize=(3 * len(phases_view), 3.2))
for ax, phase in zip(axes, phases_view):
    mesh = plot_sky_view(ax, phi, lam, w_hot, phase, vmin=0, vmax=1)
fig.suptitle('Hot-region weight, as seen by the observer', y=1.05)
fig.colorbar(mesh, ax=axes, shrink=0.7, label='$w_{hot}$', pad=0.02)

## 10. Wind field and line-of-sight velocity

`v_zonal` (Section 4's solid rotation + jet) and the day-to-night speed field are each
phase-independent, planet-frame quantities. The full-sphere map of `v_zonal` first, in the
same (longitude, latitude) coordinates used throughout this notebook so far -- the
day-to-night term is deliberately *not* part of this map (see Section 4: it is not zonal),
so unlike an earlier version of this section, there is no sharp break here anymore.

In [ ]:
v_zonal = kernel.zonal_wind_field(phi, lam)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.pcolormesh(np.degrees(lam), np.degrees(phi), v_zonal / 1e3, shading='auto', cmap='coolwarm')
ax.set_xlabel('Longitude [deg] (0 = substellar)')
ax.set_ylabel('Latitude [deg]')
ax.set_title('Zonal wind speed (rotation + jet only) [km/s] -- full sphere, planet frame')
fig.colorbar(im, ax=ax, label='km/s')
fig.tight_layout()

**The day-to-night term needs its own projection.** Its most direct path from the substellar to the antistellar point is tangent to the great circle through both -- *not* tangent to a circle of latitude, except exactly on the equator (an earlier version of this class got this wrong, using the zonal direction everywhere with a `sign(lambda)` flip; caught by inspecting a phase=0.5 sky view, where it produced an unphysical east/west split instead of the expected center-to-limb, radially-symmetric pattern). Writing `psi` for the angular distance from the substellar point ($\cos\psi = \cos\varphi\cos\lambda$), the corrected projection is:

$$v_{\rm los}^{\rm d2n}(\lambda,\varphi) = v_{\rm d2n}(\lambda,\varphi) \cdot \frac{\cos\psi \, \cos\varphi \, \cos(\lambda-\lambda_{\rm obs}) - \cos\lambda_{\rm obs}}{\sin\psi}$$

(`los_velocity_from_day_night`; full derivation in that function's docstring). A quick consistency check: on the equator this reduces to ${\rm sign}(\sin\lambda)$ times `los_velocity_from_zonal`'s own projection -- i.e. the old, equator-only-correct formula, recovered as the special case it always should have been.

Sky views of `v_zonal`, and of the two LOS velocity contributions side by side, make the fix very visible. At exactly secondary eclipse (`phase=0.5`, full dayside centered -- the case requested explicitly), `v_zonal`'s LOS projection gives the usual east/west (redshift/blueshift) split; the day-to-night term instead gives a **radially symmetric, single-signed** pattern growing from the disk center toward the limb -- gas flowing away from the substellar point is, viewed face-on, moving away from the observer in every direction at once, with no preferred east or west. The left/right split for the day-to-night term only appears away from exact eclipse (rightmost panel), once the sub-observer point drifts off the substellar meridian.

In [ ]:
day_night_speed = kernel.day_night_speed_field(phi)

lam_obs_eclipse = phase_to_lam_obs(0.5)
lam_obs_later = phase_to_lam_obs(0.65)

v_los_zonal_eclipse = los_velocity_from_zonal(v_zonal, phi, lam, lam_obs_eclipse)
v_los_dn_eclipse = los_velocity_from_day_night(day_night_speed, phi, lam, lam_obs_eclipse)
v_los_dn_later = los_velocity_from_day_night(day_night_speed, phi, lam, lam_obs_later)

vmax = np.nanmax(np.abs([v_los_zonal_eclipse, v_los_dn_eclipse, v_los_dn_later])) / 1e3

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
mesh = plot_sky_view(axes[0], phi, lam, v_los_zonal_eclipse / 1e3, phase=0.5,
                      cmap='coolwarm', vmin=-vmax, vmax=vmax)
axes[0].set_title('$v_{los}$ from rotation+jet\nphase=0.5: usual E/W split', fontsize=9)
plot_sky_view(axes[1], phi, lam, v_los_dn_eclipse / 1e3, phase=0.5,
              cmap='coolwarm', vmin=-vmax, vmax=vmax)
axes[1].set_title('$v_{los}$ from day-to-night\nphase=0.5: radial, single-signed', fontsize=9)
plot_sky_view(axes[2], phi, lam, v_los_dn_later / 1e3, phase=0.65,
              cmap='coolwarm', vmin=-vmax, vmax=vmax)
axes[2].set_title('$v_{los}$ from day-to-night\nphase=0.65: E/W split emerges', fontsize=9)
fig.colorbar(mesh, ax=axes, shrink=0.8, pad=0.02, label='km/s')

## 11. Resulting kernels across orbital phase

`get_ker(phase)` returns `[kernel_hot, kernel_cold]`, each carrying only the *geometric* (visibility x projected-area x solid-angle) weight of its region — the actual brightness contrast between hot and cold is deliberately left to the caller (same convention as `CitrusRotationKernel`, see the class docstring), so these two kernels are meant to be convolved with two different region spectra and summed downstream (e.g. via `model_sequence.combine_regions_with_kernel`). Using `kernel.plot_kernel` for each panel also throws in the instrument-degraded version for free, instead of writing this loop by hand.

In [ ]:
phases = [0.0, 0.15, 0.25, 0.5, 0.75, 0.9]
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)

for ax, phase in zip(axes.ravel(), phases):
    kernel.plot_kernel(phase, ax=ax, n_os=5)
    ax.set_title(f'phase = {phase}', fontsize=9)
    ax.get_legend().remove()

axes[0, 0].legend(fontsize=7)
for ax in axes[:, 0]:
    ax.set_ylabel('Kernel')
fig.tight_layout()

## 12. Sanity check against the analytic uniform-disk kernel

Shrinking the hotspot down to a negligible size and disabling the jet/day-night flow should make `kernel_hot + kernel_cold` reproduce `SolidRotationKernel`'s closed-form $\sqrt{1-(v/v_{\rm eq})^2}$ profile at any phase, since both are then describing the exact same physical situation (a uniformly-bright, solid-body-rotating disk) computed two different ways — this directly checks the sign convention derived in Section 3 and the correctness of the numerical grid/histogram machinery.

In [ ]:
trivial_kernel = HotspotWindRotationKernel(
    pl_rad, omega_rot, resolution,
    hotspot_lon=0 * u.deg, hotspot_half_lon=1 * u.deg, hotspot_half_lat=1 * u.deg,
    n_lat=241, n_lon=481,
)
solid_kernel = SolidRotationKernel(pl_rad, omega_rot, resolution)
v_grid_solid, ker_solid = solid_kernel.get_ker(n_os=5)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for phase, ax in zip([0.0, 0.3], axes):
    v_grid, (ker_hot, ker_cold) = trivial_kernel.get_ker(phase=phase, n_os=5)
    combined = ker_hot + ker_cold
    ker_solid_interp = np.interp(v_grid, v_grid_solid, ker_solid, left=0, right=0)
    ker_solid_interp /= ker_solid_interp.sum()
    ax.plot(v_grid / 1e3, combined, label='HotspotWindRotationKernel (numerical)')
    ax.plot(v_grid / 1e3, ker_solid_interp, '--', label='SolidRotationKernel (analytic)')
    max_rel_diff = np.abs(combined - ker_solid_interp).max() / combined.max()
    ax.set_title(f'phase={phase}, max diff = {max_rel_diff:.1%} of peak')
    ax.set_xlabel('v [km/s]')
    ax.legend(fontsize=8)
axes[0].set_ylabel('Kernel')
fig.tight_layout()
print('Residual-level agreement (a few % near the disk edge, where the analytic profile',
      'has a vertical tangent, is expected grid/histogram discretization noise -- tighten',
      'n_lat/n_lon/n_os to shrink it further).')

## 13. Effect of the superrotating jet

Adding a strong equatorial jet broadens the combined kernel (more disk area now moves at a higher zonal speed).

In [ ]:
kernel_nojet = HotspotWindRotationKernel(
    pl_rad, omega_rot, resolution,
    hotspot_lon=20 * u.deg, hotspot_half_lon=35 * u.deg, hotspot_half_lat=25 * u.deg,
    n_lat=181, n_lon=361,
)
kernel_jet = HotspotWindRotationKernel(
    pl_rad, omega_rot, resolution,
    hotspot_lon=20 * u.deg, hotspot_half_lon=35 * u.deg, hotspot_half_lat=25 * u.deg,
    jet_delta_omega=2 * np.pi / (0.4 * u.day).to('s'), jet_half_lat=15 * u.deg,
    n_lat=181, n_lon=361,
)

fig, ax = plt.subplots(figsize=(7, 4))
for label, k in [('no jet', kernel_nojet), ('with jet', kernel_jet)]:
    v_grid, (ker_hot, ker_cold) = k.get_ker(phase=0.5, n_os=5)
    ax.plot(v_grid / 1e3, ker_hot + ker_cold, label=label)
ax.set_xlabel('v [km/s]')
ax.set_ylabel('Combined kernel')
ax.set_title('Jet broadening (phase 0.5, dayside/hotspot visible)')
ax.legend()
fig.tight_layout()

## 14. Effect of the day-to-night flow — a net redshift at secondary eclipse

Section 10's sky views already showed why: at exactly secondary eclipse (`phase=0.5`), the day-to-night term's LOS projection is radially symmetric around the disk center and **single-signed everywhere** on the visible disk (growing from 0 at the very center to its full speed at the limb) -- there is no east/west cancellation to begin with, because there is no east/west asymmetry at that exact phase. The whole disk-integrated line is therefore net **redshifted** (gas is, on average, flowing from the visible dayside into the hidden nightside -- receding from the observer), not just broadened. This qualitatively matches what high-resolution emission-spectroscopy studies of ultra-hot Jupiters predict/report for day-to-night wind signatures near secondary eclipse (and is the emission-geometry analogue of the transit terminator-wind signatures in the reference PDF this notebook is built around).

In [ ]:
kernel_nodn = HotspotWindRotationKernel(
    pl_rad, omega_rot, resolution,
    hotspot_lon=20 * u.deg, hotspot_half_lon=35 * u.deg, hotspot_half_lat=25 * u.deg,
    n_lat=181, n_lon=361,
)
kernel_dn = HotspotWindRotationKernel(
    pl_rad, omega_rot, resolution,
    hotspot_lon=20 * u.deg, hotspot_half_lon=35 * u.deg, hotspot_half_lat=25 * u.deg,
    day_night_speed=3000 * u.m / u.s, day_night_lat_onset=30 * u.deg,
    n_lat=181, n_lon=361,
)

fig, ax = plt.subplots(figsize=(7, 4))
for label, k in [('no day-night flow', kernel_nodn), ('with day-night flow', kernel_dn)]:
    v_grid, (ker_hot, ker_cold) = k.get_ker(phase=0.5, n_os=5)
    combined = ker_hot + ker_cold
    mean_v = np.sum(combined * v_grid)
    ax.plot(v_grid / 1e3, combined, label=f'{label} (mean v = {mean_v:.0f} m/s)')
ax.set_xlabel('v [km/s]')
ax.set_ylabel('Combined kernel')
ax.set_title('Day-to-night flow at secondary eclipse (phase 0.5)')
ax.legend(fontsize=8)
fig.tight_layout()

## 15. One-stop diagnostics: `kernel.show()`

Sections 8-14 built each diagnostic figure by hand, one piece at a time, to explain where it comes from. Day to day, `HotspotWindRotationKernel.show()` bundles the same figures -- regions, LOS velocity, the three wind components side by side, and the native/degraded kernel -- into a single call, exactly like `CitrusRotationKernel.show()`. The individual panels (`plot_regions`, `plot_los_velocity`, `plot_velocity_components`, `plot_kernel`) are also methods in their own right, each usable on its own (pass an `ax` to drop one into a bigger custom figure) -- `show()` is just the four of them assembled.

In [ ]:
_ = kernel.show(phase=0.5)

## 16. Status and further reading

`HotspotWindRotationKernel` now lives in `starships.spectrum`, alongside every other rotation kernel, with the sanity checks from this notebook mirrored as `tests/unit/test_spectrum.py::TestHotspotWindRotationKernel` (`pytest tests/unit/test_spectrum.py -k HotspotWind`).

- **Grid resolution**: `n_lat`/`n_lon` (surface grid) and `n_os` (velocity-grid oversampling) trade accuracy for speed. The defaults here are generous for exploration; for a retrieval, benchmark the smallest grid that keeps the sanity check in Section 12 within the desired tolerance.
- **Using it in a retrieval**: exactly like the citrus example in `model_sequence.combine_regions_with_kernel` -- convolve the *native* (undegraded) hot- and cold-region model spectra with `kernel_hot`/`kernel_cold` respectively, and combine with `theta_dict['spec_scale']` weights (the actual hot/cold brightness contrast is set there, by using physically different temperature profiles for the two regions -- the kernel itself only carries the geometric visibility weight). Wiring it up as a `get_ker_file` hook (same shape as the `CitrusRotationKernel` example already documented in the retrieval yaml) is the next thing to validate against an actual retrieval run.
- **The module-level pieces** (`equal_area_latlon_grid`, `bin_weighted_velocities` especially) are reusable beyond this specific kernel -- e.g. a numerical cross-check of `CitrusRotationKernel` itself, or a future transit/transmission terminator-ring kernel built the same way as the reference PDF's Appendix A, but without needing the whole integral to stay analytic.
- **A genuinely independent wind field for the cold region** (rather than sharing the hot region's) is the most promising extension identified so far, found while applying this kernel to a real system's secondary-peak Kp-Vsys structure (`starships_analysis/KELT-20b/`) -- not yet built.